In [1]:
import os
import shutil
from google.colab import drive
drive.mount('/content/drive')

# Define model name and paths
model_name = "llama3.1-8b" # change this
drive_folder = "/content/drive/MyDrive/Models/finetunes/" + model_name + "/"
drive_gguf_path = drive_folder + "Meta-Llama-3.1-8B-Instruct.Q8_0" + ".gguf" # and this
dataset = "hamd_valid_dataset.json" # and this
drive_dataset_path = "/content/drive/MyDrive/Models/" + dataset

local_gguf_path = "/content/" + model_name + ".gguf"
local_dataset_path = "/content/" + dataset

# Copy GGUF file to local runtime
print(f"Copying model from Google Drive to local runtime...")
if not os.path.exists(local_gguf_path):
    shutil.copy2(drive_gguf_path, local_gguf_path)

# Copy dataset to local runtime
print(f"Copying dataset from Google Drive to local runtime...")
if os.path.exists(drive_dataset_path):
    shutil.copy2(drive_dataset_path, local_dataset_path)
else:
    print("Warning: Dataset not found in Drive directory.")

print("Copy complete!")

Mounted at /content/drive
Copying model from Google Drive to local runtime...
Copying dataset from Google Drive to local runtime...
Copy complete!


In [2]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:13 https://ppa.launchpadcontent.net/gra

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [4]:
# Create the Modelfile pointing to the local file
modelfile_content = f"FROM {local_gguf_path}\n"
with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Create the model in Ollama
!ollama create {model_name} -f /content/Modelfile

# List available models to verify
!ollama list


NAME                  ID              SIZE      MODIFIED               
llama3.1-8b:latest    c45c95bf0696    8.5 GB    Less than a second ago    


In [5]:
%pip install openai scikit-learn pandas

In [6]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [7]:
SYSTEM_PROMPT = """
You are a HAM-D (Hamilton Depression Rating Scale, 21-item) survey scoring assistant.
Your job is to extract answers for the HAM-D-21 based on a given conversation transcript.
The HAM-D-21 is a clinician-administered assessment that measures the severity of depression over the past week.

Each item below lists its description and its allowed score range. Different items use different scales:

q1  Depressed mood (sadness, hopelessness, helplessness, worthlessness)        scale 0-4
q2  Feelings of guilt (self-reproach, ideas of guilt, delusions of guilt)       scale 0-4
q3  Suicide (feels life not worth living, wishes dead, ideation, attempts)      scale 0-4
q4  Insomnia early (difficulty falling asleep)                                  scale 0-2
q5  Insomnia middle (waking during the night, restless sleep)                   scale 0-2
q6  Insomnia late (waking in early hours and unable to fall back asleep)        scale 0-2
q7  Work and activities (loss of interest, fatigue, decreased productivity)     scale 0-4
q8  Retardation (slowness of thought, speech, movement) - usually observed      scale 0-4
q9  Agitation (fidgeting, hand-wringing, hair-pulling) - usually observed       scale 0-4
q10 Anxiety - psychic (subjective tension, worry, irritability, fears)          scale 0-4
q11 Anxiety - somatic (GI, cardiovascular, respiratory, sweating, headaches)    scale 0-4
q12 Somatic symptoms - gastrointestinal (loss of appetite, heavy stomach)       scale 0-2
q13 Somatic symptoms - general (heaviness, fatigue, backache, muscle aches)     scale 0-2
q14 Genital symptoms (loss of libido, menstrual disturbances)                   scale 0-2
q15 Hypochondriasis (preoccupation with bodily health, conviction of illness)   scale 0-4
q16 Loss of weight (by history or measured)                                     scale 0-2
q17 Insight (recognition of being depressed/ill)                                scale 0-2
q18 Diurnal variation (whether mood is worse at a particular time of day)       scale 0-2
q19 Depersonalization and derealization (feelings of unreality, dreamlike)      scale 0-4
q20 Paranoid symptoms (suspiciousness, ideas of reference, persecution)         scale 0-4
q21 Obsessional and compulsive symptoms (intrusive thoughts, rituals)           scale 0-2

General severity guide for each scale:
0 = absent / none
1 = mild / occasional / doubtful
2 = moderate / definite
3 = severe (only for 0-4 items)
4 = very severe / incapacitating (only for 0-4 items)

Specific scoring guidance for tricky items:
- q1: 0=absent, 1=indicated only on questioning, 2=spontaneously reported verbally, 3=communicated nonverbally (facial expression, voice, tearfulness), 4=patient reports virtually only these feelings.
- q3: 0=absent, 1=feels life not worth living, 2=wishes were dead, 3=suicidal ideas or gestures, 4=attempts at suicide.
- q4/q5/q6: 0=no difficulty, 1=occasional/mild, 2=nightly/severe.
- q7: 0=no difficulty, 1=thoughts of incapacity/fatigue, 2=loss of interest in activities, 3=decreased productivity, 4=stopped working due to illness.
- q8 and q9 are clinician-observed; if not evident from the transcript, score 0.
- q15: 0=not present, 1=self-absorption (bodily), 2=preoccupation with health, 3=frequent complaints/requests for help, 4=hypochondriacal delusions.
- q16: 0=no weight loss, 1=probable weight loss associated with illness, 2=definite weight loss.
- q17: 0=acknowledges being depressed and ill, 1=acknowledges illness but attributes cause to bad food/climate/overwork/virus/need for rest, 2=denies being ill at all.
- q18: 0=no variation, 1=mild variation worse in AM or PM, 2=severe variation.
- q20: 0=none, 1=suspicious, 2=ideas of reference, 3=delusions of reference and persecution, 4=hallucinations, persecutory.

Topic order in the conversation (use this as a guide; the assistant typically asks about these in roughly this order):
mood -> crying -> guilt / self-blame -> punishment -> suicide -> sleep onset (q4) -> sleep middle (q5) -> sleep early morning (q6) -> work / activities (q7) -> anxiety / worry (q10) -> physical anxiety symptoms (q11) -> appetite (q12, infer q16) -> energy / heaviness / aches (q13) -> sex (q14) -> health concerns (q15) -> diurnal variation (q18) -> unreal / dreamlike (q19) -> paranoid (q20) -> obsessive / compulsive (q21) -> insight (q17)

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value within that item's allowed range ("0"-"4" for 0-4 items, "0"-"2" for 0-2 items).
Rules:
- Match each topic in the conversation to the correct q number based on its description above (not by response position).
- Score each item based solely on what the user said in the transcript.
- For q8 and q9, score 0 unless the user clearly describes psychomotor retardation or agitation.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape, do not include comments inside the json:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "q8": "0",
  "q9": "0",
  "q10": "0",
  "q11": "0",
  "q12": "0",
  "q13": "0",
  "q14": "0",
  "q15": "0",
  "q16": "0",
  "q17": "0",
  "q18": "0",
  "q19": "0",
  "q20": "0",
  "q21": "0"
}
"""
MODEL = model_name

- Use a json dataset of HAM-D-21 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [8]:
# Load the HAM-D-21 dataset
import json

with open('hamd_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

SUCCESS: Number of conversations matches number of scores
Total conversations: 10


In [9]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

Train: 7, Test: 3


In [10]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [11]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

Generating 1/7
Generating 2/7
Generating 3/7
Generating 4/7
Generating 5/7
Generating 6/7
Generating 7/7
Generating 1/3
Generating 2/3
Generating 3/3


In [12]:
# Parse json scores
def parse_scores(raw_scores):
    parsed = []
    for s in raw_scores:
        try:
            parsed.append(json.loads(s))
        except json.JSONDecodeError:
            print(f'Failed to parse: {s}')
            parsed.append({f'q{i}': '0' for i in range(1, 22)})
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [13]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    cols = [f'q{i}' for i in range(1, 22)]
    pred_df = pred_df[cols]
    expected_df = expected_df[cols]

    mse = mean_squared_error(
        convert_scores_to_array(expected_df),
        convert_scores_to_array(pred_df)
    )
    print(f'{name} MSE: {mse}')

    display(expected_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    ))

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')

Train MSE: 0.6462585034013606


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        4         4        3         3        2         3        2         2   
1        2         2        1         2        0         0        2         1   
2        4         4        3         3        2         3        0         0   
3        2         2        0         0        0         0        0         2   
4        4         4        3         4        2         3        2         2   
5        3         4        1         3        2         0        0         2   
6        1         0        0         0        0         0        0         0   

        q5            ...      q17                q18                q19  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        2         2  ...        1         1        1         2        0   
1        0         0  ...        0         1        1         1        0   
2        1         2  ...        0         1        1         2        2   
3        1         2  ...        2         1        2         2        0   
4        2         2  ...        0         1        2         2        0   
5        0         0  ...        0         1        0         0        1   
6        0         0  ...        0         0        0         0        0   

                 q20                q21            
  Predicted Expected Predicted Expected Predicted  
0         0        0         0        0         0  
1         0        0         0        0         0  
2         2        2         0        2         0  
3         0        0         0        0         0  
4         0        0         0        0         0  
5         2        2         0        0         0  
6         0        0         0        0         0  

[7 rows x 42 columns]

Test MSE: 0.8095238095238095


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        3         3        2         3        1         0        2         2   
1        4         4        3         4        3         4        2         2   
2        3         2        2         3        0         0        2         1   

        q5            ...      q17                q18                q19  \
  Expected Predicted  ... Expected Predicted Expected Predicted Expected   
0        1         2  ...        1         1        2         2        0   
1        2         2  ...        2         1        0         2        2   
2        1         0  ...        1         1        0         0        1   

                 q20                q21            
  Predicted Expected Predicted Expected Predicted  
0         0        0         0        0         0  
1         4        0         0        0         0  
2         2        0         0        0         1  

[3 rows x 42 columns]